# 01 — Learn teacher trajectories

Train multiple teacher/expert models and save their optimisation trajectories.
Change only the configuration cell to switch between **eICU**, **MIMIC-III in-hospital mortality**, and **MIMIC-III phenotyping**.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "btm").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from btm.loaders import DATASET_CHOICES, load_dataset
from btm.models import default_net_type, get_model
from btm.utils import *


## Configuration


In [ ]:
DATASET = "eicu"          # "eicu", "mimic3_ihm", "mimic3_ph"
NET_TYPE = None           # None -> dataset default (DNN for eICU, TCN2 for MIMIC)
OPTIM = "sgd"             # "sgd", "gsam", or "adam"
NUM_EXPERTS = 50
TRAIN_BATCH = 256
SEED = 42

assert DATASET in DATASET_CHOICES
NET_TYPE = NET_TYPE or default_net_type(DATASET)
set_seed(SEED)
device = get_device()
print("device:", device)
print("dataset/model/optim:", DATASET, NET_TYPE, OPTIM)


## Load the selected dataset


In [ ]:
bundle = load_dataset(
    DATASET,
    # data_root=ROOT / "data",
    data_root='../../data',
    train_batch=TRAIN_BATCH,
    pre_process="std",
)

train_loader = bundle.train_loader
val_loader = bundle.val_loader
test_loader = bundle.test_loader

print("task:", bundle.task)
print("features:", bundle.num_features, "output_dim:", bundle.output_dim)
print("batches:", len(train_loader), len(val_loader), len(test_loader))


## Train expert trajectories


In [ ]:
lr, momentum, num_epochs = get_training_hyperparameters(
    DATASET, OPTIM, stage="teacher"
)
print(f"lr={lr}, momentum={momentum}, epochs={num_epochs}")

model_factory = lambda: get_model(
    DATASET, NET_TYPE, device=device, output_dim=bundle.output_dim
)

trajectories = train_teacher_trajectories(
    model_factory=model_factory,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    task=bundle.task,
    device=device,
    num_experts=NUM_EXPERTS,
    optim=OPTIM,
    num_epochs=num_epochs,
    lr=lr,
    momentum=momentum,
)


## Save


In [ ]:
paths = experiment_paths(DATASET, NET_TYPE, OPTIM, root=ROOT)
paths["trajectory_dir"].mkdir(parents=True, exist_ok=True)
torch.save(trajectories, paths["trajectory"])
print("saved:", paths["trajectory"])


## Summary


In [ ]:
test_aucs = np.asarray([r["test_auc"] for r in trajectories])
test_aprs = np.asarray([r["test_apr"] for r in trajectories])

print(f"test AUC: {test_aucs.mean():.4f} ± {test_aucs.std():.4f}")
print(f"test APR: {test_aprs.mean():.4f} ± {test_aprs.std():.4f}")


## Optional: inspect one trajectory


In [ ]:
EXPERT_IDX = 0
trajectory = trajectories[EXPERT_IDX]["trajectory"]
loss_fn = nn.BCEWithLogitsLoss()
model = model_factory()

train_losses, val_losses, val_aucs, val_aprs = [], [], [], []
for checkpoint in trajectory:
    model.load_state_dict(checkpoint)
    train_losses.append(evaluate_loss(model, train_loader, loss_fn, device, task=bundle.task))
    metrics = evaluate_model(model, val_loader, loss_fn, device, bundle.task)
    val_losses.append(metrics["loss"])
    val_aucs.append(metrics["auc"])
    val_aprs.append(metrics["apr"])

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.xlabel("checkpoint / epoch")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.show()
